<a href="https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/05_Errors_Modules_and_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Errors, modules, and files

| | |
|---|---|
| Time | about 50 minutes |
| Prerequisites | Notebooks 00–04 |
| Environment | Google Colab, or any Python 3 notebook |
| Thread | Bike-share trips in Indian cities — the same story builds toward the capstone |

## How to use this notebook

1. Click **Open in Colab**, then use **File → Save a copy in Drive**.
2. Run cells **from top to bottom**. If names go missing, use **Runtime → Restart session** and **Run all**.
3. When you see **Predict**, decide the output before you run.
4. Practice cells include a working answer so **Run all** never fails (needed for class and CI). To learn, comment out the answer, rewrite it, then run the check.
5. Hints sit under each practice. Clean solutions are at the end.

Concept sections include diagrams of the idea, not only the code. They show as pictures in Colab and as Mermaid on GitHub.

## What you will be able to do

- Tell syntax, runtime, and logic errors apart
- Catch a specific exception and raise a clear one
- Import from the standard library
- Read and write JSON and CSV with `pathlib`


## Recap warm-up


In [ ]:
def double(n):
    return n * 2

print(double(21))


## Three kinds of errors

| Kind | When you see it | Example |
|---|---|---|
| Syntax | before the cell really runs | missing colon, bad indent |
| Runtime | while running | `int("bike")`, `1/0`, missing key |
| Logic | it runs, the answer is wrong | averaging with the wrong list |

Catch only errors you expect. Bare `except:` hides bugs. After you convert data, `assert` the shape you promised.

![Syntax, runtime, and logic errors show up at different times](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgc3ludGF4WyJTeW50YXgmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXSAtLT4gYmVmb3JlUnVuWyJGYWlscyBiZWZvcmUgdGhlIGNlbGwgcnVucyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgcnVudGltZUVyclsiUnVudGltZSZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdIC0tPiB3aGlsZVJ1blsiRmFpbHMgd2hpbGUgcnVubmluZyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgbG9naWNFcnJbIkxvZ2ljJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0gLS0-IHdyb25nQW5zd2VyWyJSdW5zIGJ1dCB0aGUgYW5zd2VyIGlzIHdyb25nJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  syntax[Syntax] --> beforeRun[Fails before the cell runs]
  runtimeErr[Runtime] --> whileRun[Fails while running]
  logicErr[Logic] --> wrongAnswer[Runs but the answer is wrong]
```

</details>

![Catch only the failure you expect, then continue](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgdHJ5QmxvY2tbInRyeSZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdIC0tPiBva1BhdGhbInJldHVybiBpbnQodGV4dCkmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHRyeUJsb2NrIC0tPiBiYWRQYXRoWyJWYWx1ZUVycm9yJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBiYWRQYXRoIC0tPiBjYXRjaE5vbmVbInJldHVybiBOb25lJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart LR
  tryBlock[try] --> okPath["return int(text)"]
  tryBlock --> badPath[ValueError]
  badPath --> catchNone[return None]
```

</details>


In [ ]:
# Syntax: caught so Run all continues
try:
    compile("if True print('missing colon')", "<cell>", "exec")
except SyntaxError as exc:
    print("syntax:", type(exc).__name__)

# Runtime
raw = "not a number"
try:
    int(raw)
except ValueError as exc:
    print("runtime:", type(exc).__name__, exc)

# Logic: the cell runs, the answer is still wrong
km = [8.0, 4.0]
wrong_mean = sum(km) / 3
right_mean = sum(km) / len(km)
print("logic bug mean", wrong_mean, "correct mean", right_mean)

def require_positive(hours):
    if hours < 0:
        raise ValueError("hours must be >= 0")
    return hours

print(require_positive(2))
try:
    require_positive(-1)
except ValueError as exc:
    print(exc)

assert require_positive(2) == 2


## Modules and files

`import math` loads a library. Colab files live in a **temporary** folder (`/content` on Colab). Restarting the runtime can delete them. This notebook writes under `Path.cwd() / "_course_data"` so it also works in GitHub Actions.

Always open files with a `with` block so they close even if something fails. Prefer `pathlib.Path` over raw `"/tmp/file.csv"` strings.

![JSON is a dict written as text, then read back as a dict](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgcGF5bG9hZFsiZGljdCZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdIC0tPiBkdW1wc1sianNvbi5kdW1wcyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgZHVtcHMgLS0-IHdyaXRlRmlsZVsiUGF0aC53cml0ZV90ZXh0Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICB3cml0ZUZpbGUgLS0-IHJlYWRGaWxlWyJQYXRoLnJlYWRfdGV4dCZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgcmVhZEZpbGUgLS0-IGxvYWRzWyJqc29uLmxvYWRzJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBsb2FkcyAtLT4gZGljdEJhY2tbImRpY3QmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQ==)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart LR
  payload[dict] --> dumps["json.dumps"]
  dumps --> writeFile["Path.write_text"]
  writeFile --> readFile["Path.read_text"]
  readFile --> loads["json.loads"]
  loads --> dictBack[dict]
```

</details>


In [ ]:
import csv
import json
import math
from pathlib import Path

print(round(math.pi, 4))

DATA_DIR = Path.cwd() / "_course_data"
DATA_DIR.mkdir(exist_ok=True)

payload = {"city": "Mumbai", "trips": 3}
json_path = DATA_DIR / "rider.json"
json_path.write_text(json.dumps(payload), encoding="utf-8")
loaded = json.loads(json_path.read_text(encoding="utf-8"))
print(loaded)

csv_path = DATA_DIR / "trips.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "city"])
    writer.writeheader()
    writer.writerow({"id": 1, "city": "Pune"})

with csv_path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))
print(rows)


## Practice 05.1


Write `parse_int(text)` that returns the integer, or `None` if conversion fails.


In [ ]:
# Practice 05.1
def parse_int(text):
    try:
        return int(text)
    except (TypeError, ValueError):
        return None

print(parse_int("42"), parse_int("nope"))

# Check 05.1
assert parse_int("42") == 42
assert parse_int("nope") is None
print("Practice 05.1 passed.")


<details>
<summary>Hint 05.1</summary>

Wrap `int(text)` in `try`. Catch `ValueError` and `TypeError`, then `return None`.

</details>


## Practice 05.2


Write a dict to `DATA_DIR / "practice.json"` and read it back.


In [ ]:
# Practice 05.2
practice_path = DATA_DIR / "practice.json"
record = {"id": 10, "city": "Bengaluru"}
practice_path.write_text(json.dumps(record), encoding="utf-8")
restored = json.loads(practice_path.read_text(encoding="utf-8"))
print(restored)

# Check 05.2
assert restored == record
print("Practice 05.2 passed.")


<details>
<summary>Hint 05.2</summary>

`json.dumps` turns a dict into text. `json.loads` reverses it.

</details>


## Practice 05.3


Parse this CSV text into a list of dictionaries with integer `id` values.


In [ ]:
# Practice 05.3
import io

csv_text = "id,city\n1,Mumbai\n2,Pune\n"
parsed = []
reader = csv.DictReader(io.StringIO(csv_text))
for row in reader:
    parsed.append({"id": int(row["id"]), "city": row["city"]})
print(parsed)

# Check 05.3
assert parsed == [{"id": 1, "city": "Mumbai"}, {"id": 2, "city": "Pune"}]
print("Practice 05.3 passed.")


<details>
<summary>Hint 05.3</summary>

`csv.DictReader` yields rows. Convert `row["id"]` with `int`.

</details>


## Common mistakes

- Catching every exception and continuing silently.
- Opening a file and never closing it (use `with`).
- Assuming Colab files survive a runtime restart. They often do not.


## Recap


Name the error, catch only what you expect, and keep data in files you control. JSON and CSV are the usual hand-off formats before pandas.


## Check your understanding

<details>
<summary>int("bike") is which kind of problem?</summary>

A runtime `ValueError`. The line is legal Python; it fails while running.

</details>

<details>
<summary>Why open files with a with block?</summary>

The file is closed even if the cell fails halfway through.

</details>


## Solutions


<details>
<summary>Solution 05.1</summary>

```python
def parse_int(text):
    try:
        return int(text)
    except (TypeError, ValueError):
        return None
```

</details>


<details>
<summary>Solution 05.2</summary>

```python
practice_path.write_text(json.dumps(record), encoding="utf-8")
restored = json.loads(practice_path.read_text(encoding="utf-8"))
```

</details>


<details>
<summary>Solution 05.3</summary>

```python
parsed = []
for row in csv.DictReader(io.StringIO(csv_text)):
    parsed.append({"id": int(row["id"]), "city": row["city"]})
```

</details>


## Navigation

- Previous: [04_Functions_Scope_and_Comprehensions.ipynb](https://github.com/ialimustufa/python-crash-course/blob/master/notebooks/04_Functions_Scope_and_Comprehensions.ipynb) · [Open in Colab](https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/04_Functions_Scope_and_Comprehensions.ipynb)
- Next: [06_Python_Data_Workflows_for_ML.ipynb](https://github.com/ialimustufa/python-crash-course/blob/master/notebooks/06_Python_Data_Workflows_for_ML.ipynb) · [Open in Colab](https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/06_Python_Data_Workflows_for_ML.ipynb)
- Course home: [README.md](https://github.com/ialimustufa/python-crash-course/blob/master/README.md)
